파일 시스템 vs DB
1. 조회 및 집계 속도 (데이터가 늘어날수록 체감되는 차이)
2. 데이터 무결성과 관계(JOIN) (잘못된 주문 방지, 연관 데이터 결합)
3. 동시 수정(재고 차감 등) 안전성

## 1. 한눈에 보는 비교: 파일 시스템(CSV/JSON) vs. 관계형 DB(MySQL)

| 비교 항목 | 파일 시스템 (CSV / JSON / TXT) | 데이터베이스 (MySQL 등 DBMS) |
| :--- | :--- | :--- |
| **데이터 검색** | 보통 파일 전체를 읽고 반복문/필터링 수행 | 인덱스(Index), 쿼리 최적화기(Optimizer) 활용 |
| **연관 데이터 결합** | 코드에서 직접 딕셔너리/리스트로 매칭(`JOIN` 구현 필요) | `JOIN` 연산으로 테이블 간 관계를 선언적으로 결합 |
| **데이터 무결성** | 오타, 누락, 없는 상품 ID 참조가 쉽게 발생 | `PRIMARY KEY`, `FOREIGN KEY`, 타입 제약으로 차단 |
| **동시 접근/수정** | 동시 쓰기 시 덮어쓰기·파일 손상 위험 | 트랜잭션(ACID)과 락(Lock)으로 동시성 제어 |
| **적합한 상황** | 소규모 설정, 로그 백업, 단순 데이터 교환 | 다중 사용자 서비스, 정합성이 중요한 업무 데이터 |

In [ ]:
# 2. 비교 실험용 가상 주문 데이터(50,000건) 생성: CSV vs SQLite/MySQL
import csv
import json
import os
import random
import sqlite3
import time

NUM_ORDERS = 50000
cities = ['서울', '남원', '한양', '구월산', '백령도']
levels = ['Silver', 'Gold', 'VIP', 'VVIP']
categories = ['가전', '식품', '잡화']

random.seed(42)

orders_data = []
for oid in range(1, NUM_ORDERS + 1):
    u_id = random.randint(1, 1000)
    p_id = random.randint(1, 200)
    amount = random.randint(1, 10)
    price = random.choice([800, 1500, 45000, 150000, 1200000, 1300000])
    city = random.choice(cities)
    level = random.choice(levels)
    category = random.choice(categories)
    orders_data.append({
        'o_id': oid,
        'u_id': u_id,
        'p_id': p_id,
        'amount': amount,
        'price': price,
        'city': city,
        'level': level,
        'category': category
    })

# 1) CSV 저장
csv_path = 'orders_sample.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=orders_data[0].keys())
    writer.writeheader()
    writer.writerows(orders_data)

# 2) JSON 저장
json_path = 'orders_sample.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(orders_data, f, ensure_ascii=False)

# 3) SQLite DB 저장 + 인덱스 생성
db_path = 'mart_bench.db'
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
cur = conn.cursor()
cur.execute('''
CREATE TABLE orders (
    o_id INTEGER PRIMARY KEY,
    u_id INTEGER,
    p_id INTEGER,
    amount INTEGER,
    price INTEGER,
    city TEXT,
    level TEXT,
    category TEXT
)
''')
cur.executemany('''
INSERT INTO orders VALUES (:o_id, :u_id, :p_id, :amount, :price, :city, :level, :category)
''', orders_data)
cur.execute('CREATE INDEX idx_orders_level_cat ON orders(level, category)')
conn.commit()

print(f"생성 완료! 주문 건수: {NUM_ORDERS:,}건")
print(f"CSV 크기 : {os.path.getsize(csv_path)/1024:.1f} KB")
print(f"JSON 크기: {os.path.getsize(json_path)/1024:.1f} KB")
print(f"DB 크기  : {os.path.getsize(db_path)/1024:.1f} KB")

In [ ]:
# 3. 속도 비교: 'VIP' 고객의 '가전' 카테고리 총 매출 집계하기

# A. CSV 파일을 매번 읽고 필터링/집계하는 방식
t0 = time.perf_counter()
csv_total = 0
csv_count = 0
with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['level'] == 'VIP' and row['category'] == '가전':
            csv_total += int(row['price']) * int(row['amount'])
            csv_count += 1
csv_elapsed = (time.perf_counter() - t0) * 1000

# B. JSON 파일을 파싱하고 필터링/집계하는 방식
t0 = time.perf_counter()
with open(json_path, 'r', encoding='utf-8') as f:
    loaded = json.load(f)
json_total = sum(r['price'] * r['amount'] for r in loaded if r['level'] == 'VIP' and r['category'] == '가전')
json_count = sum(1 for r in loaded if r['level'] == 'VIP' and r['category'] == '가전')
json_elapsed = (time.perf_counter() - t0) * 1000

# C. DB 쿼리(SELECT + WHERE + SUM) 실행 방식
t0 = time.perf_counter()
cur.execute('''
SELECT COUNT(*), SUM(price * amount)
FROM orders
WHERE level = 'VIP' AND category = '가전'
''')
db_count, db_total = cur.fetchone()
db_elapsed = (time.perf_counter() - t0) * 1000

print(f"[CSV]  건수={csv_count}, 총액={csv_total:,}원 | 소요시간: {csv_elapsed:.2f} ms")
print(f"[JSON] 건수={json_count}, 총액={json_total:,}원 | 소요시간: {json_elapsed:.2f} ms")
print(f"[DB]   건수={db_count}, 총액={db_total:,}원 | 소요시간: {db_elapsed:.2f} ms")

1. 인덱스(Index, 색인): 책 뒤의 '찾아보기'처럼 조건(`WHERE`)에 맞는 데이터를 빠르게 찾도록 돕는 자료구조(보통 B-Tree 계열)를 사용해 전체 데이터를 빠르게 찾음.
2. 쿼리 옵티마이저(Query Optimizer): 우리가 `SELECT ... WHERE ... GROUP BY`로 '무엇을 원하는지'만 적으면, DB 엔진이 가장 적은 비용으로 결과를 내는 실행 계획을 자동으로 실행.
3. 메모리 버퍼 풀(Buffer Pool) 캐싱: 자주 조회되는 데이터 페이지를 디스크가 아닌 빠른 메모리(RAM)에 올려두고 재사용.
4. 무결성과 트랜잭션 보장: 잘못된 참조(`FOREIGN KEY` 위반)를 막고, 여러 작업이 섞여도 데이터가 꼬이지 않게 보호.

In [ ]:
# 4. '좋음'의 핵심: 파일은 허용하지만 DB는 막아주는 데이터 무결성
conn.execute('PRAGMA foreign_keys = ON;')
cur.execute('CREATE TABLE IF NOT EXISTS products_ref (p_id INTEGER PRIMARY KEY, p_name TEXT)')
cur.execute('CREATE TABLE IF NOT EXISTS orders_ref (o_id INTEGER PRIMARY KEY, p_id INTEGER REFERENCES products_ref(p_id))')
cur.execute("INSERT OR IGNORE INTO products_ref VALUES (1, '갤럭시 S24')")
conn.commit()
cur.execute("SELECT * FROM products_ref WHERE p_id = 1")
print("상품 등록 확인 ->", cur.fetchone())

# 파일(CSV/JSON)이라면 없는 상품(999번) 주문도 그냥 기록 되어버림.
fake_csv_row = {'o_id': 99999, 'p_id': 999, 'note': '존재하지 않는 상품인데 CSV에는 그냥 들어감'}
print("파일 방식 예시 ->", fake_csv_row)

# DB는 외래 키 제약조건으로 잘못된 참조를 차단.
try:
    cur.execute("INSERT INTO orders_ref VALUES (1, 999)")
    conn.commit()
except sqlite3.IntegrityError as e:
    print("DB 차단 성공 -> 존재하지 않는 상품(999) 주문 거부:", e)

conn.close()